# 벡터 덧셈 연산을 순수 CUDA C++ 커널로 작성하고 파이썬에서 호출

> 본 노트북은 SYLLABUS.md에 기반하여 자동 생성된 뼈대(Skeleton) 파일입니다. 상세한 이론, 수식 및 코드는 추가로 구현되어야 합니다.

## 1. 개요 및 학습 목표
이 노트북에서는 해당 주제에 대한 핵심 개념을 다룹니다.

## 2. 핵심 이론 및 수학적 원리
- 수식 및 상세한 동작 원리를 여기에 기록합니다.

## 3. 실습 코드 구현
아래 셀을 통해 파이썬 및 관련 프레임워크 코드를 직접 작성하고 실행해 볼 수 있습니다.

In [ ]:
# 실습을 위한 기본 라이브러리 임포트
import tensorflow as tf
import numpy as np

print(f"TensorFlow Version: {tf.__version__}")


---
## Q1: 벡터 덧셈 — NumPy vs Python 루프 벤치마크

CUDA 커널의 필요성을 체감하기 위해, 먼저 순수 Python 루프와 NumPy 벡터화 연산의 속도 차이를 직접 측정해 보세요.

**요구사항:**
- 크기 `N = 10_000_000`인 float32 배열 A, B를 생성하세요.
- 순수 Python `for`문과 NumPy `a + b` 두 방식으로 벡터 합산을 수행하고 `time.time()`으로 소요 시간을 측정하세요.
- 두 방식의 결과가 동일한지 검증하고, 속도 차이(배율)를 출력하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np
import time

N = 10_000_000
np.random.seed(42)
a = np.random.randn(N).astype(np.float32)
b = np.random.randn(N).astype(np.float32)

# Python 루프 (작은 N으로)
N_small = 100_000
t0 = time.time()
c_loop = [a[i] + b[i] for i in range(N_small)]
loop_time = time.time() - t0

# NumPy
t0 = time.time()
c_numpy = a + b
numpy_time = time.time() - t0

print(f"Python 루프 ({N_small:,}개): {loop_time*1000:.2f} ms")
print(f"NumPy ({N:,}개): {numpy_time*1000:.2f} ms")
print(f"NumPy가 동일 크기 기준 ~{(loop_time/N_small)/(numpy_time/N):.0f}x 빠름")
```
</details>

---
## Q2: 스레드 그리드 / 블록 구성 계산기

CUDA 커널에서 스레드 수와 블록 수를 올바르게 계산하는 함수를 작성하세요.

**요구사항:**
- `calc_grid_block(N, threads_per_block=256)` 함수를 정의하세요.
- `num_blocks`는 `ceil(N / threads_per_block)`으로 계산하세요.
- N = 1000, 10_000, 1_000_000, 10_000_000 에 대해 각각 블록 수와 전체 스레드 수를 출력하세요.
- 전체 스레드 수가 N을 넘는 이유(패딩)를 설명하는 주석을 추가하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import math

def calc_grid_block(N, threads_per_block=256):
    # 데이터 N개를 커버하려면 블록이 올림 나눗셈만큼 필요
    num_blocks = math.ceil(N / threads_per_block)
    total_threads = num_blocks * threads_per_block
    return num_blocks, total_threads

test_sizes = [1_000, 10_000, 1_000_000, 10_000_000]
threads = 256

print(f"{'N':>12} {'블록 수':>10} {'전체 스레드':>14} {'패딩 스레드':>12}")
print('-' * 52)
for N in test_sizes:
    blocks, total = calc_grid_block(N, threads)
    padding = total - N
    print(f"{N:>12,} {blocks:>10,} {total:>14,} {padding:>12,}")
```
</details>

---
## Q3: CuPy로 벡터 덧셈 가속 (GPU 있는 경우) / NumPy 타임라인 시뮬레이션

CUDA 커널의 개념을 CuPy(GPU가 있는 경우) 또는 NumPy로 시뮬레이션하여, 커널이 어떻게 데이터를 처리하는지 직접 체험하세요.

**요구사항:**
- GPU가 없는 환경에선 `numpy`를 `cupy`처럼 사용하는 시뮬레이션으로 대체하세요.
- N = 10_000_000 배열 덧셈을 수행하고 소요 시간을 측정하세요.
- 커널 개념 설명: 배열을 `num_blocks`개의 블록으로 나누어 각 블록을 병렬로 처리하는 구조를 NumPy 슬라이싱으로 시뮬레이션하세요.

In [ ]:
# TODO: 코드를 직접 작성해 보세요.

<details>
<summary>💡 정답 보기</summary>

```python
import numpy as np
import time

try:
    import cupy as cp
    xp = cp
    print("CuPy 사용 (GPU 가속)")
except ImportError:
    xp = np
    print("CuPy 없음 → NumPy로 시뮬레이션")

N = 10_000_000
a = xp.random.randn(N).astype(xp.float32)
b = xp.random.randn(N).astype(xp.float32)

# 블록 단위 처리 시뮬레이션
threads_per_block = 256
num_blocks = (N + threads_per_block - 1) // threads_per_block
c = xp.zeros(N, dtype=xp.float32)

t0 = time.time()
for block_idx in range(min(num_blocks, 100)):  # 시뮬레이션으로 처음 100블록만
    start = block_idx * threads_per_block
    end = min(start + threads_per_block, N)
    c[start:end] = a[start:end] + b[start:end]
blocked_time = time.time() - t0

t0 = time.time()
c_full = a + b  # 실제 벡터 연산
full_time = time.time() - t0

print(f"블록 루프 (처음 100블록): {blocked_time*1000:.2f} ms")
print(f"전체 벡터 연산: {full_time*1000:.2f} ms")
print("\n→ GPU에서는 이 블록들이 동시(병렬)로 처리됩니다!")
```
</details>